# Pandas: practical tabular data analysis

Pandas is the main tool for working with tables in this course. This notebook shows you a common workflow: load data, inspect it, clean it, transform it, summarize it, and prepare it for analysis.

The dataset contains **69 entirely fictional AAI students**. It was generated for artificially and it is not real University of Bayreuth data.

## 1. Imports and portable data loading

The conventional alias for Pandas is `pd`. `pathlib.Path` helps us find the CSV whether Jupyter starts in the repository root or in `00_Setup_Solutions`.

In [ ]:
from pathlib import Path
import pandas as pd


candidate_paths = [
  parent / "data" / "aai_student_data.csv"
  for parent in (Path.cwd(), *Path.cwd().parents)
]


data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
  raise FileNotFoundError("Could not find data/aai_student_data.csv in the repository.")

print(f"Loading: {data_path.resolve()}")

## 2. Series and DataFrames

A `Series` is one labeled column. A `DataFrame` is a table of aligned Series. You can construct both directly from Python objects.

In [2]:
scores = pd.Series([99, 69, 3], index=["Julika", "Leopold", "Victor"], name="exam_score")
print(scores)
print(scores.mean())

tiny_frame = pd.DataFrame({
  "student": ["Ada", "Grace", "Margaret"],
  "score": [67, 69, 99],
  "passed": [True, True, True],
})
display(tiny_frame)

Julika     99
Leopold    69
Victor      3
Name: exam_score, dtype: int64
57.0


,student,score,passed
0,Ada,67,True
1,Grace,69,True
2,Margaret,99,True


In [3]:
# In jupyter notebooks, you can also just type the variable name to display it:
tiny_frame

,student,score,passed
0,Ada,67,True
1,Grace,69,True
2,Margaret,99,True


### Load a CSV

`read_csv()` is one of the functions you will use most often. The returned object is a DataFrame.

In [4]:
students = pd.read_csv(data_path)
students.head()

,student_id,study_program,python_experience,weekly_study_hours,attendance_percent,assignments_completed,coffee_cups_per_week,exam_score,passed,tutorial_group,enrollment_date
0,AAI001,Business Administration,Intermediate,6.0,90,8,1,70.2,True,T1,2026-04-07
1,AAI002,Business Administration,Advanced,11.3,100,8,10,80.8,True,T2,2026-04-08
2,AAI003,Philosophy,Basic,9.4,71,9,1,70.5,True,T3,2026-04-09
3,AAI004,Mathematics,Intermediate,7.6,79,8,14,60.7,True,T1,2026-04-10
4,AAI005,Computer Science,Basic,5.0,59,10,0,47.2,False,T2,2026-04-11


## 3. First inspection

Never begin an analysis by trusting the file name. Check rows, columns, types, missing values, and plausible ranges first.

In [5]:
print("Shape:", students.shape)
print("Columns:", students.columns.tolist())
print("Index:", students.index)
students.info()

Shape: (69, 11)
Columns: ['student_id', 'study_program', 'python_experience', 'weekly_study_hours', 'attendance_percent', 'assignments_completed', 'coffee_cups_per_week', 'exam_score', 'passed', 'tutorial_group', 'enrollment_date']
Index: RangeIndex(start=0, stop=69, step=1)
<class 'pandas.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             69 non-null     str    
 1   study_program          69 non-null     str    
 2   python_experience      51 non-null     str    
 3   weekly_study_hours     66 non-null     float64
 4   attendance_percent     69 non-null     int64  
 5   assignments_completed  69 non-null     int64  
 6   coffee_cups_per_week   69 non-null     int64  
 7   exam_score             69 non-null     float64
 8   passed                 69 non-null     bool   
 9   tutorial_group         69 non-null     str    
 10  enr

In [6]:
students.describe(include="all")

,student_id,study_program,python_experience,weekly_study_hours,attendance_percent,assignments_completed,coffee_cups_per_week,exam_score,passed,tutorial_group,enrollment_date
count,69,69,51,66.000000,69.000000,69.000000,69.000000,69.000000,69,69,69
unique,69,5,3,NaN,NaN,NaN,NaN,NaN,2,3,12
top,AAI001,Sports Economics,Basic,NaN,NaN,NaN,NaN,NaN,True,T1,2026-04-07
freq,1,18,24,NaN,NaN,NaN,NaN,NaN,46,23,6
mean,NaN,NaN,NaN,6.501515,80.246377,6.927536,6.768116,55.968116,NaN,NaN,NaN
std,NaN,NaN,NaN,2.838661,14.074806,1.888968,4.069777,11.358432,NaN,NaN,NaN
min,NaN,NaN,NaN,0.500000,35.000000,3.000000,0.000000,31.600000,NaN,NaN,NaN
25%,NaN,NaN,NaN,5.000000,70.000000,6.000000,3.000000,47.500000,NaN,NaN,NaN
50%,NaN,NaN,NaN,6.300000,79.000000,7.000000,7.000000,56.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,8.075000,91.000000,8.000000,10.000000,63.300000,NaN,NaN,NaN


In [7]:
students.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
student_id,69,69,AAI001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
study_program,69,5,Sports Economics,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
python_experience,51,3,Basic,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weekly_study_hours,66.0,NaN,NaN,NaN,6.501515,2.838661,0.5,5.0,6.3,8.075,14.8
attendance_percent,69.0,NaN,NaN,NaN,80.246377,14.074806,35.0,70.0,79.0,91.0,100.0
assignments_completed,69.0,NaN,NaN,NaN,6.927536,1.888968,3.0,6.0,7.0,8.0,10.0
coffee_cups_per_week,69.0,NaN,NaN,NaN,6.768116,4.069777,0.0,3.0,7.0,10.0,15.0
exam_score,69.0,NaN,NaN,NaN,55.968116,11.358432,31.6,47.5,56.0,63.3,88.8
passed,69,2,True,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tutorial_group,69,3,T1,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
display(students.head(3))
print(students.tail(3))
print(students.sample(3, random_state=420))

,student_id,study_program,python_experience,weekly_study_hours,attendance_percent,assignments_completed,coffee_cups_per_week,exam_score,passed,tutorial_group,enrollment_date
0,AAI001,Business Administration,Intermediate,6.0,90,8,1,70.2,True,T1,2026-04-07
1,AAI002,Business Administration,Advanced,11.3,100,8,10,80.8,True,T2,2026-04-08
2,AAI003,Philosophy,Basic,9.4,71,9,1,70.5,True,T3,2026-04-09


   student_id     study_program python_experience  weekly_study_hours  \
66     AAI067       Mathematics             Basic                 7.7   
67     AAI068        Philosophy          Advanced                 3.5   
68     AAI069  Computer Science             Basic                 3.7   

    attendance_percent  assignments_completed  coffee_cups_per_week  \
66                  81                      7                     8   
67                  87                      8                     0   
68                  73                      4                     2   

    exam_score  passed tutorial_group enrollment_date  
66        56.0    True             T1      2026-04-13  
67        60.6    True             T2      2026-04-14  
68        43.5   False             T3      2026-04-15  
   student_id     study_program python_experience  weekly_study_hours  \
36     AAI037  Sports Economics             Basic                11.9   
67     AAI068        Philosophy          Advanced   

### Solution: inspect

Display seven randomly selected rows reproducibly and calculate the number of missing values per column.

**Explanation:** `sample()` selects seven rows, while `random_state=420` makes the selection reproducible. `isna()` creates a boolean missing-value mask and `sum()` counts the `True` values column by column.

In [9]:
### --- Start Solution --- ###
inspection_sample = students.sample(7, random_state=420)
missing_per_column = students.isna().sum()
### --- End Solution --- ###

print(inspection_sample)
print(missing_per_column)

   student_id            study_program python_experience  weekly_study_hours  \
36     AAI037         Sports Economics             Basic                11.9   
67     AAI068               Philosophy          Advanced                 3.5   
55     AAI056         Sports Economics               NaN                 5.8   
30     AAI031               Philosophy          Advanced                 3.5   
56     AAI057         Sports Economics             Basic                 9.2   
68     AAI069         Computer Science             Basic                 3.7   
52     AAI053  Business Administration               NaN                 7.6   

    attendance_percent  assignments_completed  coffee_cups_per_week  \
36                  70                      9                     9   
67                  87                      8                     0   
55                  79                     10                    14   
30                  64                      5                     2   
56  

## 4. Selecting rows and columns

Use brackets for one or more columns, `.loc` for label-based selection, and `.iloc` for position-based selection.

In [10]:
score_series = students["exam_score"]
small_table = students[["student_id", "study_program", "exam_score"]]

print(score_series.head())
print(small_table.head())

0    70.2
1    80.8
2    70.5
3    60.7
4    47.2
Name: exam_score, dtype: float64
  student_id            study_program  exam_score
0     AAI001  Business Administration        70.2
1     AAI002  Business Administration        80.8
2     AAI003               Philosophy        70.5
3     AAI004              Mathematics        60.7
4     AAI005         Computer Science        47.2


In [11]:
print(students.loc[0:2, ["student_id", "exam_score"]]) # Labels 0, 1, and 2 are included. Human-logical indexing.
print(students.iloc[0:3, [0, 7]]) # Positions 0, 1, and 2. Stop is excluded. Python-logical indexing.

  student_id  exam_score
0     AAI001        70.2
1     AAI002        80.8
2     AAI003        70.5
  student_id  exam_score
0     AAI001        70.2
1     AAI002        80.8
2     AAI003        70.5


Boolean masks filter rows. Parenthesize each comparison when combining masks with `&` or `|`.

In [12]:
hard_working = students[
  (students["weekly_study_hours"] >= 8)
  & (students["assignments_completed"] >= 8)
]

business_or_math = students[
  students["study_program"].isin(["Business Administration", "Mathematics"])
]

print(hard_working[["student_id", "weekly_study_hours", "assignments_completed"]].head())
print(business_or_math["study_program"].value_counts())

   student_id  weekly_study_hours  assignments_completed
1      AAI002                11.3                      8
2      AAI003                 9.4                      9
24     AAI025                10.0                      9
31     AAI032                 9.7                      8
33     AAI034                 8.3                      8
study_program
Business Administration    12
Mathematics                10
Name: count, dtype: int64


### Solution: filter

Select the ID, program, and score of students in tutorial group `T2` who scored at least `69`.

**Explanation:** The two boolean conditions form the row mask. `.loc` applies that mask and selects only the three requested columns in one operation.

In [13]:
### --- Start Solution --- ###
selected_students = students.loc[
  (students["tutorial_group"] == "T2") & (students["exam_score"] >= 69),
  ["student_id", "study_program", "exam_score"],
]
### --- End Solution --- ###

print(selected_students)

   student_id            study_program  exam_score
1      AAI002  Business Administration        80.8
22     AAI023         Sports Economics        72.2
25     AAI026               Philosophy        72.9


## 5. Sorting, renaming, and derived columns

In [14]:
top_students = students.sort_values(
  by=["exam_score", "weekly_study_hours"],
  ascending=[False, True],
)
top_students[["student_id", "exam_score", "weekly_study_hours"]].head()

,student_id,exam_score,weekly_study_hours
11,AAI012,88.8,14.8
1,AAI002,80.8,11.3
24,AAI025,78.8,10.0
56,AAI057,73.1,9.2
25,AAI026,72.9,7.1


In [15]:
analysis = students.rename(columns={"exam_score": "score"}).copy()
analysis["study_minutes"] = analysis["weekly_study_hours"] * 60
analysis["score_per_assignment"] = analysis["score"] / analysis["assignments_completed"].replace(0, pd.NA)
analysis[["student_id", "score", "study_minutes", "score_per_assignment"]].head()

,student_id,score,study_minutes,score_per_assignment
0,AAI001,70.2,360.0,8.775000
1,AAI002,80.8,678.0,10.100000
2,AAI003,70.5,564.0,7.833333
3,AAI004,60.7,456.0,7.587500
4,AAI005,47.2,300.0,4.720000


Use `.assign()` for a readable transformation chain. Each step returns a new DataFrame.

In [16]:
enriched = students.assign(
  attendance_fraction=lambda frame: frame["attendance_percent"] / 100,
  passed_label=lambda frame: frame["passed"].map({True: "passed", False: "not passed"}),
)
enriched[["student_id", "attendance_fraction", "passed_label"]].head()

,student_id,attendance_fraction,passed_label
0,AAI001,0.90,passed
1,AAI002,1.00,passed
2,AAI003,0.71,passed
3,AAI004,0.79,passed
4,AAI005,0.59,not passed


## 6. Missing values, duplicates, and types

The fictional data intentionally contains a few missing values. Decide whether to keep, remove, or impute them based on the analysis---not because `.dropna()` was the first method you remembered.

In [17]:
students.isna().sum()

student_id                0
study_program             0
python_experience        18
weekly_study_hours        3
attendance_percent        0
assignments_completed     0
coffee_cups_per_week      0
exam_score                0
passed                    0
tutorial_group            0
enrollment_date           0
dtype: int64

In [18]:
clean_students = students.copy()
median_hours = clean_students["weekly_study_hours"].median()
clean_students["weekly_study_hours"] = clean_students["weekly_study_hours"].fillna(median_hours)
clean_students["python_experience"] = clean_students["python_experience"].fillna("Unknown")

print(clean_students.isna().sum())

student_id               0
study_program            0
python_experience        0
weekly_study_hours       0
attendance_percent       0
assignments_completed    0
coffee_cups_per_week     0
exam_score               0
passed                   0
tutorial_group           0
enrollment_date          0
dtype: int64


Create duplicates in memory so the source CSV stays clean. `duplicated()` identifies them. `drop_duplicates()` removes them.

In [19]:
with_duplicate = pd.concat([students, students.iloc[[0]]], ignore_index=True)
print("Duplicate rows:", with_duplicate.duplicated().sum())

without_duplicates = with_duplicate.drop_duplicates()
print(without_duplicates.shape)

Duplicate rows: 1
(69, 11)


Convert types explicitly. `errors='coerce'` turns invalid numeric text into missing values. Parse date text with `to_datetime()`. Categories can save memory and communicate a finite set of labels.

In [20]:
typed_students = clean_students.copy()
typed_students["enrollment_date"] = pd.to_datetime(typed_students["enrollment_date"])
typed_students["tutorial_group"] = typed_students["tutorial_group"].astype("category")
typed_students["exam_score"] = pd.to_numeric(typed_students["exam_score"], errors="coerce")

print(typed_students.dtypes)
print(typed_students["enrollment_date"].dt.day_name().head())

student_id                          str
study_program                       str
python_experience                   str
weekly_study_hours              float64
attendance_percent                int64
assignments_completed             int64
coffee_cups_per_week              int64
exam_score                      float64
passed                             bool
tutorial_group                 category
enrollment_date          datetime64[us]
dtype: object
0      Tuesday
1    Wednesday
2     Thursday
3       Friday
4     Saturday
Name: enrollment_date, dtype: str


### Safe assignment

Avoid chained assignment such as `students[mask]['passed'] = True`. It may modify a temporary object. Use `.loc[rows, columns]` on an explicit copy.

In [21]:
adjusted_students = students.copy()
high_attendance = adjusted_students["attendance_percent"] >= 95
adjusted_students.loc[high_attendance, "attendance_label"] = "excellent"
adjusted_students.loc[~high_attendance, "attendance_label"] = "regular"
adjusted_students[["attendance_percent", "attendance_label"]].head()

,attendance_percent,attendance_label
0,90,regular
1,100,excellent
2,71,regular
3,79,regular
4,59,regular


### Solution: clean

Create a copy named `exercise_clean` and fill missing Python experience with `"Not reported"`.

**Explanation:** The transformation is performed on a copy so that the original DataFrame remains available for comparison. The filled Series is assigned back to the column explicitly.

In [22]:
exercise_clean = students.copy()

### --- Start Solution --- ###
exercise_clean["python_experience"] = exercise_clean["python_experience"].fillna("Not reported")
### --- End Solution --- ###

print(exercise_clean["python_experience"].value_counts(dropna=False))

python_experience
Basic           24
Intermediate    18
Not reported    18
Advanced         9
Name: count, dtype: int64


## 7. Strings, mapping, and `apply`

The `.str` accessor applies vectorized string operations. `map()` is ideal for value-to-value mappings. Prefer vectorized expressions. Use `apply()` when row logic is genuinely clearer.

In [23]:
string_demo = students[["student_id", "study_program"]].copy()
string_demo["program_lower"] = string_demo["study_program"].str.lower()
string_demo["mentions_science"] = string_demo["study_program"].str.contains("Science", case=False)
string_demo.head()

,student_id,study_program,program_lower,mentions_science
0,AAI001,Business Administration,business administration,False
1,AAI002,Business Administration,business administration,False
2,AAI003,Philosophy,philosophy,False
3,AAI004,Mathematics,mathematics,False
4,AAI005,Computer Science,computer science,True


In [24]:
experience_order = {
  "None": 0,
  "Basic": 1,
  "Intermediate": 2,
  "Advanced": 3,
}
mapped_students = students.copy()
mapped_students["experience_level"] = mapped_students["python_experience"].map(experience_order)
mapped_students[["python_experience", "experience_level"]].head(10)

,python_experience,experience_level
0,Intermediate,2.0
1,Advanced,3.0
2,Basic,1.0
3,Intermediate,2.0
4,Basic,1.0
5,NaN,NaN
6,Intermediate,2.0
7,Intermediate,2.0
8,Basic,1.0
9,Basic,1.0


In [25]:
def performance_label(row: pd.Series) -> str:
  if row["exam_score"] >= 80 and row["assignments_completed"] >= 8:
    return "strong and consistent"
  if row["exam_score"] >= 50:
    return "passed"
  return "needs another round"


labelled_students = students.copy()
labelled_students["performance_label"] = labelled_students.apply(performance_label, axis=1)
labelled_students["performance_label"].value_counts()

performance_label
passed                   45
needs another round      23
strong and consistent     1
Name: count, dtype: int64

## 8. Grouping and aggregation

The split-apply-combine pattern groups rows, calculates summaries, and combines the results.

In [26]:
program_summary = (
  students
  .groupby("study_program", as_index=False)
  .agg(
    students=("student_id", "count"),
    average_score=("exam_score", "mean"),
    median_study_hours=("weekly_study_hours", "median"),
    pass_rate=("passed", "mean"),
  )
  .sort_values("average_score", ascending=False)
)

program_summary.round(2)

,study_program,students,average_score,median_study_hours,pass_rate
1,Computer Science,12,56.88,7.05,0.50
3,Philosophy,17,56.85,6.00,0.71
2,Mathematics,10,55.82,7.65,0.90
4,Sports Economics,18,55.18,6.70,0.61
0,Business Administration,12,55.11,5.30,0.67


In [27]:
students["python_experience"].value_counts(dropna=False, normalize=True).mul(100).round(1)

python_experience
Basic           34.8
Intermediate    26.1
NaN             26.1
Advanced        13.0
Name: proportion, dtype: float64

### Solution: aggregate

For each tutorial group, calculate the number of students and the median exam score.

**Explanation:** `groupby()` creates one group per tutorial. Named aggregation produces clear output-column names for the student count and median score.

In [28]:
### --- Start Solution --- ###
tutorial_summary = (
  students
  .groupby("tutorial_group", as_index=False)
  .agg(
    students=("student_id", "count"),
    median_exam_score=("exam_score", "median"),
  )
)
### --- End Solution --- ###

print(tutorial_summary)

  tutorial_group  students  median_exam_score
0             T1        23               56.0
1             T2        23               57.2
2             T3        23               53.9


## 9. Combining and reshaping tables

`merge()` joins tables using key columns. Validate the expected relationship when possible.

In [29]:
program_support = pd.DataFrame({
  "study_program": [
    "Computer Science",
    "Mathematics",
    "Business Administration",
    "Sports Economics",
    "Philosophy",
  ],
  "support_contact": ["CS Lab", "Math Lab", "Business Lab", "Sports Lab", "Humanities Lab"],
})

students_with_support = students.merge(
  program_support,
  on="study_program",
  how="left",
  validate="many_to_one",
)
students_with_support[["student_id", "study_program", "support_contact"]].head()

,student_id,study_program,support_contact
0,AAI001,Business Administration,Business Lab
1,AAI002,Business Administration,Business Lab
2,AAI003,Philosophy,Humanities Lab
3,AAI004,Mathematics,Math Lab
4,AAI005,Computer Science,CS Lab


`concat()` stacks compatible objects. Here we split and recombine rows only to demonstrate the operation.

In [30]:
first_half = students.iloc[:35]
second_half = students.iloc[35:]
recombined = pd.concat([first_half, second_half], ignore_index=True)
print(recombined.shape)
print(recombined["student_id"].is_unique)

(69, 11)
True


Pivot tables summarize combinations of categories. `melt()` converts wide columns into a longer, tidy representation.

In [31]:
score_pivot = pd.pivot_table(
  students,
  index="study_program",
  columns="tutorial_group",
  values="exam_score",
  aggfunc="mean",
).round(1)
score_pivot

tutorial_group,T1,T2,T3
study_program,,,
Business Administration,55.5,59.1,49.0
Computer Science,60.1,52.1,57.9
Mathematics,53.7,52.0,61.9
Philosophy,59.6,55.9,55.4
Sports Economics,50.8,64.7,53.8


In [32]:
long_metrics = students.melt(
  id_vars=["student_id"],
  value_vars=["weekly_study_hours", "attendance_percent", "exam_score"],
  var_name="metric",
  value_name="value",
)
long_metrics.head()

,student_id,metric,value
0,AAI001,weekly_study_hours,6.0
1,AAI002,weekly_study_hours,11.3
2,AAI003,weekly_study_hours,9.4
3,AAI004,weekly_study_hours,7.6
4,AAI005,weekly_study_hours,5.0


### Solution: reshape

Create a pivot table of mean `weekly_study_hours` by study program and tutorial group.

**Explanation:** The pivot table places programs on the rows and tutorial groups on the columns. Each cell contains the mean weekly study time for that combination.

In [33]:
### --- Start Solution --- ###
study_hours_pivot = pd.pivot_table(
  students,
  index="study_program",
  columns="tutorial_group",
  values="weekly_study_hours",
  aggfunc="mean",
).round(2)
### --- End Solution --- ###

print(study_hours_pivot)

tutorial_group             T1    T2    T3
study_program                            
Business Administration  5.20  7.20  2.88
Computer Science         7.48  8.00  7.87
Mathematics              6.26  8.25  9.30
Philosophy               4.55  5.90  7.28
Sports Economics         6.40  7.77  5.62


## 10. Compact exploratory workflow

A first pass should answer: Is the shape plausible? Which data is missing? What are the distributions and groups? Which relationships deserve a plot or a model? Descriptive patterns are not automatically causal conclusions.

In [34]:
eda_summary = {
  "rows": len(students),
  "columns": students.shape[1],
  "missing_values": int(students.isna().sum().sum()),
  "mean_score": round(students["exam_score"].mean(), 2),
  "pass_rate_percent": round(students["passed"].mean() * 100, 1),
  "most_common_program": students["study_program"].mode().iat[0],
}
eda_summary

{'rows': 69,
 'columns': 11,
 'missing_values': 21,
 'mean_score': np.float64(55.97),
 'pass_rate_percent': np.float64(66.7),
 'most_common_program': 'Sports Economics'}

In [35]:
numeric_relationships = students[
  ["weekly_study_hours", "attendance_percent", "assignments_completed", "exam_score"]
].corr(numeric_only=True)
numeric_relationships.round(2)

,weekly_study_hours,attendance_percent,assignments_completed,exam_score
weekly_study_hours,1.00,0.06,0.13,0.58
attendance_percent,0.06,1.00,-0.07,0.24
assignments_completed,0.13,-0.07,1.00,0.32
exam_score,0.58,0.24,0.32,1.00


### Exporting results

`to_csv()` writes a DataFrame.

```python
program_summary.to_csv("program_summary.csv", index=False)
```

Also explore `read_excel()`, `to_excel()`, `read_parquet()`, and database connectors when your project needs them.

## Takeaway

A reliable Pandas workflow is:

1. Load and inspect.
2. Select and validate.
3. Clean explicitly.
4. Transform without chained assignment.
5. Group, combine, and reshape.
6. Communicate assumptions and limitations.

References: [Pandas getting started](https://pandas.pydata.org/docs/getting_started/index.html) and [Pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html).

Next: **00_04_Numpy.ipynb**.